# Ablation Study — Statistical Summary & Inference

This notebook takes the raw cross-validation results from the ablation experiment and produces:

- aggregated summary statistics (mean/std + confidence intervals)
- performance deltas vs the baseline (full feature set)
- statistical significance tests (paired tests across folds with multiple-comparison correction)
- a feature impact ranking (importance by performance drop)

**Input:** `ablation_results.csv`  
**Outputs:**
- `ablation_summary_stats.csv`
- `ablation_significance_tests.csv`
- `ablation_feature_importance.csv`

## 1) Load results and validate format

We expect one row per **fold × model variant** with the following columns:

- identifiers: `model_variant`, `features_removed`, `fold`
- metrics: `r2`, `adjusted_r2`, `rmse`, `mae`, `mse`, `mape`

The baseline variant is named **`Baseline`** and should appear once per fold.

In [2]:
import numpy as np
import pandas as pd
from scipy import stats

INPUT_CSV = "ablation_results.csv"

OUT_STATS = "ablation_summary_stats.csv"
OUT_SIG = "ablation_significance_tests.csv"
OUT_IMP = "ablation_feature_importance.csv"

METRICS = ["r2", "adjusted_r2", "rmse", "mae", "mse", "mape"]
BASELINE_NAME = "Baseline"

CONF_LEVEL = 0.95
ALPHA = 0.05
CORRECTION = "holm"     
PRIMARY_METRIC = "mae" 


In [3]:
results_df = pd.read_csv(INPUT_CSV)
results_df["features_removed"] = results_df["features_removed"].fillna("none")

print("Shape:", results_df.shape)
print("Columns:", results_df.columns.tolist())
print("# folds:", results_df["fold"].nunique(), "| folds:", sorted(results_df["fold"].unique()))
print("Variants:", results_df["model_variant"].unique().tolist())

baseline_rows = (results_df["model_variant"] == BASELINE_NAME).sum()
print("Baseline rows:", baseline_rows)

coverage = results_df.groupby("model_variant")["fold"].nunique()
print("Fold coverage (min/max):", int(coverage.min()), int(coverage.max()))
results_df.head()

Shape: (25, 11)
Columns: ['model_variant', 'features_removed', 'fold', 'r2', 'adjusted_r2', 'rmse', 'mae', 'mse', 'mape', 'delta_r2', 'delta_rmse']
# folds: 5 | folds: [1, 2, 3, 4, 5]
Variants: ['Baseline', '- engine_power', '- mileage', '- age', '- brand_encoded']
Baseline rows: 5
Fold coverage (min/max): 5 5


,model_variant,features_removed,fold,r2,adjusted_r2,rmse,mae,mse,mape,delta_r2,delta_rmse
0,Baseline,none,1,0.662425,0.623845,2160.239640,1669.407526,4.666635e+06,8.655815,NaN,NaN
1,- engine_power,engine_power,1,0.608928,0.576338,2325.118700,1891.436489,5.406177e+06,9.868951,-0.053497,164.879060
2,- mileage,mileage,1,0.638999,0.608915,2233.937951,1777.622177,4.990479e+06,9.255545,-0.023426,73.698312
3,- age,age,1,0.003544,-0.079494,3711.467963,2909.134103,1.377499e+07,15.029315,-0.658881,1551.228324
4,- brand_encoded,brand_encoded,1,0.667625,0.639927,2143.536505,1662.009304,4.594749e+06,8.598064,0.005200,-16.703135


## Summary statistics and confidence intervals
We aggregate each metric across CV folds for each ablation variant and compute a t-based confidence interval for the mean.

In [11]:
def mean_ci_t(x, confidence=0.95):
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    n = len(x)
    if n < 2:
        return (np.nan, np.nan)
    mu = x.mean()
    se = stats.sem(x)
    tcrit = stats.t.ppf(1 - (1-confidence)/2, df=n-1)
    return (mu - tcrit * se, mu + tcrit * se)

rows = []
for (variant, feat_removed), g in results_df.groupby(["model_variant", "features_removed"]):
    row = {"model_variant": variant, "features_removed": feat_removed, "n_folds": g["fold"].nunique()}
    for m in METRICS:
        vals = g[m].to_numpy()
        row[f"{m}_mean"] = float(np.mean(vals))
        row[f"{m}_std"] = float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan
        ci_lo, ci_hi = mean_ci_t(vals, confidence=CONF_LEVEL)
        row[f"{m}_ci_lower"] = ci_lo
        row[f"{m}_ci_upper"] = ci_hi
        row[f"{m}_sem"] = float(stats.sem(vals)) if len(vals) > 1 else np.nan
        row[f"{m}_min"] = float(np.min(vals))
        row[f"{m}_max"] = float(np.max(vals))
    rows.append(row)

stats_df = pd.DataFrame(rows)

In [5]:
stats_df[["model_variant", "features_removed", "r2_mean", "rmse_mean", "mae_mean"]].sort_values("model_variant")

,model_variant,features_removed,r2_mean,rmse_mean,mae_mean
0,- age,age,0.033547,3837.144657,3244.118407
1,- brand_encoded,brand_encoded,0.713124,2085.853506,1653.046247
2,- engine_power,engine_power,0.642713,2334.115685,1900.018901
3,- mileage,mileage,0.700655,2123.553543,1693.636551
4,Baseline,none,0.709854,2097.205500,1662.729673


In [12]:
baseline_rows = stats_df[stats_df["model_variant"] == BASELINE_NAME]
if len(baseline_rows) != 1:
    raise ValueError(f"Expected exactly 1 baseline row, found {len(baseline_rows)}")

baseline_row = baseline_rows.iloc[0]

for m in METRICS:
    base = baseline_row[f"{m}_mean"]
    stats_df[f"delta_{m}"] = stats_df[f"{m}_mean"] - base
    stats_df[f"delta_{m}_pct"] = np.where(base != 0, 100 * stats_df[f"delta_{m}"] / base, np.nan)

stats_df[["model_variant", "features_removed", "delta_r2", "delta_rmse", "delta_mae"]].sort_values(
    "delta_mae", ascending=False
)


,model_variant,features_removed,delta_r2,delta_rmse,delta_mae
0,- age,age,-0.676307,1739.939157,1581.388734
2,- engine_power,engine_power,-0.067141,236.910186,237.289228
3,- mileage,mileage,-0.009199,26.348044,30.906878
4,Baseline,none,0.000000,0.000000,0.000000
1,- brand_encoded,brand_encoded,0.003270,-11.351994,-9.683426


## Significance tests (paired vs baseline)
Paired tests across folds comparing each ablation to the baseline, with Holm correction.

In [13]:
def adjust_pvalues(pvals, method="holm"):
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)

    if method == "none":
        return pvals
    if method == "bonferroni":
        return np.clip(pvals * m, 0, 1)

    order = np.argsort(pvals)
    ranked = pvals[order]
    adj = np.empty_like(ranked)

    for i, p in enumerate(ranked):
        adj[i] = (m - i) * p

    adj = np.maximum.accumulate(adj)
    adj = np.clip(adj, 0, 1)

    out = np.empty_like(adj)
    out[order] = adj
    return out


all_rows = []

for metric in METRICS:
    piv = results_df.pivot_table(index="fold", columns="model_variant", values=metric, aggfunc="mean")
    base = piv[BASELINE_NAME]
    variants = [v for v in piv.columns if v != BASELINE_NAME]

    rows = []
    pvals = []

    for v in variants:
        common = pd.concat([base, piv[v]], axis=1).dropna()
        diffs = common[v] - common[BASELINE_NAME]
        t_stat, p_val = stats.ttest_rel(common[BASELINE_NAME], common[v])

        rows.append({
            "metric": metric,
            "model_variant": v,
            "mean_diff_variant_minus_baseline": float(diffs.mean()),
            "n_pairs": int(len(common)),
            "t_stat": float(t_stat),
            "p_value": float(p_val),
        })
        pvals.append(p_val)

    tmp = pd.DataFrame(rows)
    tmp["p_value_adj"] = adjust_pvalues(tmp["p_value"].values, method=CORRECTION)
    tmp["significant"] = tmp["p_value_adj"] < ALPHA
    all_rows.append(tmp)

significance_df = pd.concat(all_rows, ignore_index=True).sort_values(["metric", "p_value_adj"])
significance_df


,metric,model_variant,mean_diff_variant_minus_baseline,n_pairs,t_stat,p_value,p_value_adj,significant
4,adjusted_r2,- age,-7.236851e-01,5,15.455744,0.000102,0.000409,True
5,adjusted_r2,- brand_encoded,1.252271e-02,5,-6.355662,0.003141,0.006699,True
6,adjusted_r2,- engine_power,-6.375560e-02,5,6.965683,0.002233,0.006699,True
7,adjusted_r2,- mileage,-9.844546e-04,5,0.120431,0.909949,0.909949,False
12,mae,- age,1.581389e+03,5,-13.294091,0.000185,0.000740,True
14,mae,- engine_power,2.372892e+02,5,-4.716464,0.009196,0.027587,True
13,mae,- brand_encoded,-9.683426e+00,5,2.478449,0.068325,0.136649,False
15,mae,- mileage,3.090688e+01,5,-1.089554,0.337153,0.337153,False
20,mape,- age,9.035576e+00,5,-12.554697,0.000232,0.000926,True
22,mape,- engine_power,1.310682e+00,5,-4.846737,0.008359,0.025077,True


## Feature importance (based on MAE)
Rank features by MAE increase when removed, and attach adjusted p-values.

In [14]:
importance_df = stats_df[stats_df["model_variant"] != BASELINE_NAME].copy()
importance_df["impact_score"] = importance_df["delta_mae"]

sig_mae = significance_df[significance_df["metric"] == PRIMARY_METRIC][
    ["model_variant", "p_value_adj", "significant"]
]

importance_df = importance_df.merge(sig_mae, on="model_variant", how="left")

importance_df = importance_df.sort_values("impact_score", ascending=False).reset_index(drop=True)
importance_df["rank"] = np.arange(1, len(importance_df) + 1)

importance_df[["rank", "features_removed", "impact_score", "p_value_adj", "significant"]]


,rank,features_removed,impact_score,p_value_adj,significant
0,1,age,1581.388734,0.000740,True
1,2,engine_power,237.289228,0.027587,True
2,3,mileage,30.906878,0.337153,False
3,4,brand_encoded,-9.683426,0.136649,False


In [16]:
stats_df.to_csv(OUT_STATS, index=False)
significance_df.to_csv(OUT_SIG, index=False)

importance_export = importance_df[["rank", "features_removed", "impact_score", "p_value_adj", "significant"]]
importance_export.to_csv(OUT_IMP, index=False)

print("Saved:", OUT_STATS, stats_df.shape)
print("Saved:", OUT_SIG, significance_df.shape)
print("Saved:", OUT_IMP, importance_export.shape)


Saved: ablation_summary_stats.csv (5, 57)
Saved: ablation_significance_tests.csv (24, 8)
Saved: ablation_feature_importance.csv (4, 5)


In [19]:
pd.read_csv(OUT_SIG).shape
pd.read_csv(OUT_SIG).head()


,metric,model_variant,mean_diff_variant_minus_baseline,n_pairs,t_stat,p_value,p_value_adj,significant
0,adjusted_r2,- age,-0.723685,5,15.455744,0.000102,0.000409,True
1,adjusted_r2,- brand_encoded,0.012523,5,-6.355662,0.003141,0.006699,True
2,adjusted_r2,- engine_power,-0.063756,5,6.965683,0.002233,0.006699,True
3,adjusted_r2,- mileage,-0.000984,5,0.120431,0.909949,0.909949,False
4,mae,- age,1581.388734,5,-13.294091,0.000185,0.000740,True
